# Session 2: where can the gripper go? Workspace, joint limits and redundancy
**A. Real-world problem.** Before you buy or build a pick-and-place arm you must ask: can
it reach every spot on my work table? Which spots does it lose when the motors can only turn
a limited angle? And a 3-link arm has one more joint than a point in the plane needs: what
does that extra freedom do?

**Plan.** (1) forward kinematics with cumulative angles (equations), (2) the reachable
region by sampling a grid, (3) joint limits, (4) the family of solutions, (5) design.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
LINKS = np.array([0.30, 0.25, 0.15])      # metres

## Step 1: forward kinematics for n links
With cumulative angles c_k = q_1 + ... + q_k the gripper is
    x = sum_k L_k cos(c_k),     y = sum_k L_k sin(c_k)

In [ ]:
def fk(q, links=LINKS):
    """Gripper position (x, y) in metres for joint angles q in radians."""
    c = np.cumsum(np.asarray(q, dtype=float))
    links = np.asarray(links)
    return np.array([np.sum(links * np.cos(c)), np.sum(links * np.sin(c))])


def chain(q, links=LINKS):
    """Base, elbow, wrist, gripper points (4 x 2) for plotting."""
    c = np.cumsum(np.asarray(q, dtype=float))
    steps = np.asarray(links)[:, None] * np.stack([np.cos(c), np.sin(c)], axis=1)
    return np.vstack([[0.0, 0.0], np.cumsum(steps, axis=0)])


def wrap(a):
    """Wrap an angle (or array) into (-pi, pi]."""
    return np.pi - np.mod(np.pi - np.asarray(a, dtype=float), 2 * np.pi)

## Step 2: the workspace of two links is a ring
With links L1 and L2 the distance from the base ranges from |L1 - L2| to L1 + L2, so the
reachable region is an annulus (ring) of area pi ((L1+L2)^2 - (L1-L2)^2) = 4 pi L1 L2.
We can measure an area by sampling: lay a fine grid, count cells that are reachable, and
multiply by the area of one cell.

In [ ]:
def annulus_area(r_out, r_in=0.0):
    return float(np.pi * (r_out ** 2 - r_in ** 2))


def two_link_area_grid(l1, l2, cell=0.005):
    n = int(np.ceil(2 * (l1 + l2) / cell))
    xs = -(l1 + l2) + (np.arange(n) + 0.5) * cell
    gx, gy = np.meshgrid(xs, xs)
    r = np.hypot(gx, gy)
    return float(np.count_nonzero((r >= abs(l1 - l2)) & (r <= l1 + l2)) * cell * cell)

## Step 3: a third link with the tool pointing in a chosen direction phi
If the gripper must point along phi, the wrist (end of link 2) must sit at
target - L3 (cos phi, sin phi); then the two-link ring rule applies to the wrist. For each
phi and each of the two elbow choices, two angles q1, q2 follow, and q3 = phi - q1 - q2.
Trying many phi values answers "can the gripper reach this point with ANY tool direction?".
Joint limits |q_i| <= limit throw out some of these solutions.

In [ ]:
def solutions_for_tool(target, phi, links=LINKS):
    """Both joint-angle solutions (q1, q2, q3) in radians that put the gripper on target pointing along phi."""
    l1, l2, l3 = links
    w = np.asarray(target, dtype=float) - l3 * np.array([np.cos(phi), np.sin(phi)])
    d2 = float(w @ w)
    cb = (d2 - l1 ** 2 - l2 ** 2) / (2 * l1 * l2)
    if abs(cb) > 1 + 1e-12:
        return []
    b = float(np.arccos(np.clip(cb, -1, 1)))
    out = []
    for q2 in ([b] if abs(np.sin(b)) < 1e-5 else [b, -b]):  # fully stretched or folded: one solution
        q1 = np.arctan2(w[1], w[0]) - np.arctan2(l2 * np.sin(q2), l1 + l2 * np.cos(q2))
        q1 = float(wrap(q1))
        out.append((q1, float(q2), float(wrap(phi - q1 - q2))))
    return out


def reachable_mask(x, y, limit=np.pi, links=LINKS, dphi=np.radians(5)):
    """Boolean array: which points (x, y) can be reached with every |q_i| <= limit for SOME tool direction
    and elbow choice. Same maths as solutions_for_tool, done for all points and directions at once."""
    l1, l2, l3 = links
    x = np.asarray(x, dtype=float)[..., None]
    y = np.asarray(y, dtype=float)[..., None]
    phi = -np.pi + dphi * np.arange(int(round(2 * np.pi / dphi)))
    wx, wy = x - l3 * np.cos(phi), y - l3 * np.sin(phi)
    cb = (wx ** 2 + wy ** 2 - l1 ** 2 - l2 ** 2) / (2 * l1 * l2)
    ok_ring = np.abs(cb) <= 1 + 1e-12
    b = np.arccos(np.clip(cb, -1, 1))
    good = np.zeros(np.broadcast(wx, phi).shape, dtype=bool)
    for q2 in (b, -b):
        q1 = wrap(np.arctan2(wy, wx) - np.arctan2(l2 * np.sin(q2), l1 + l2 * np.cos(q2)))
        q3 = wrap(phi - q1 - q2)
        good |= ok_ring & (np.abs(q1) <= limit + 1e-12) & (np.abs(q2) <= limit + 1e-12) & (np.abs(q3) <= limit + 1e-12)
    return good.any(axis=-1)


def reachable(target, limit=np.pi, links=LINKS, dphi=np.radians(5)):
    """True if some tool direction and elbow choice reaches target with every |q_i| <= limit."""
    return bool(reachable_mask(target[0], target[1], limit, links, dphi))


def reachable_fraction(limit_deg, links=LINKS, cell=0.02, dphi_deg=5):
    """Fraction of the full disk (radius = sum of links) that is reachable with |q_i| <= limit_deg."""
    r_max = float(np.sum(links))
    xs = -r_max + (np.arange(int(np.ceil(2 * r_max / cell))) + 0.5) * cell
    gx, gy = np.meshgrid(xs, xs)
    inside = np.hypot(gx, gy) <= r_max
    ok = reachable_mask(gx[inside], gy[inside], np.radians(limit_deg), links, np.radians(dphi_deg))
    return float(np.count_nonzero(ok) / np.count_nonzero(inside))

## Step 4: redundancy, a whole family of solutions
A point in the plane needs 2 numbers; three joints give 3. The spare one is the tool
direction phi: every phi in the reachable range gives a different pair of arm shapes for
the SAME gripper position.

In [ ]:
def solution_family(target, links=LINKS, n=72, limit=np.pi):
    """List of (phi, q1, q2, q3) for the same target over n tool directions."""
    fam = []
    for phi in np.linspace(-np.pi, np.pi, n, endpoint=False):
        for q in solutions_for_tool(target, phi, links):
            if max(abs(v) for v in q) <= limit + 1e-12:
                fam.append((float(phi),) + q)
    return fam

## Step 5: a design question
The base sits 0.20 m in front of the short side of a 0.5 m x 0.3 m table, centred on it.
Which total arm length reaches every table point? The farthest points are the far corners.

In [ ]:
def table_points(step=0.025, near=0.20, length=0.5, width=0.3):
    xs = np.arange(near, near + length + 1e-9, step)
    ys = np.arange(-width / 2, width / 2 + 1e-9, step)
    return [(float(x), float(y)) for x in xs for y in ys]


def table_coverage(links, limit_deg=180, step=0.025):
    pts = table_points(step)
    return sum(reachable(p, np.radians(limit_deg), links) for p in pts) / len(pts)

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    print("two-link annulus area, formula:", round(annulus_area(0.55, 0.05), 4),
          " grid:", round(two_link_area_grid(0.30, 0.25), 4))
    for lim in (180, 120, 90):
        print(f"limit +-{lim}: reachable fraction {reachable_fraction(lim):.3f}")
    r_max = float(np.sum(LINKS))
    cell = 0.02
    xs = -r_max + (np.arange(int(np.ceil(2 * r_max / cell))) + 0.5) * cell
    fig, axs = plt.subplots(1, 3, figsize=(10.5, 3.7), sharey=True)
    for ax, lim in zip(axs, (180, 120, 60)):
        gx, gy = np.meshgrid(xs, xs)
        m = reachable_mask(gx, gy, np.radians(lim))
        ax.plot(gx[m], gy[m], "s", ms=2, color="tab:blue")
        ax.set_aspect("equal")
        ax.grid(alpha=0.3)
        ax.set_title(f"joint limit +-{lim} deg")
        ax.set_xlabel("x (m)")
    axs[0].set_ylabel("y (m)")
    fig.tight_layout()
    fig.savefig(OUT / "workspace_limits.png", dpi=110)

    fam = solution_family((0.45, 0.20))
    fig2, ax2 = plt.subplots(figsize=(5.8, 5.2))
    for k in range(0, len(fam), 4):
        _, a, b, c = fam[k]
        pp = chain((a, b, c))
        ax2.plot(pp[:, 0], pp[:, 1], "-", lw=1.5, alpha=0.7)
    ax2.plot(0.45, 0.20, "rs")
    ax2.set_aspect("equal")
    ax2.grid(alpha=0.3)
    ax2.set_xlabel("x (m)")
    ax2.set_ylabel("y (m)")
    ax2.set_title(f"{len(fam)} arm shapes, one gripper position (0.45, 0.20) m")
    fig2.tight_layout()
    fig2.savefig(OUT / "redundancy_family.png", dpi=120)
    for links in ([0.30, 0.25, 0.15], [0.30, 0.25, 0.17], [0.30, 0.25, 0.18]):
        print(links, "table coverage:", table_coverage(links), " with +-120:", table_coverage(links, 120))

## Try it
Do the coding tasks, the README questions (F), then the checkpoint.

## Coding tasks
1. **Easy.** Print `reachable_fraction(110)` and `reachable_fraction(130)` and compare with the checkpoint values at 120 degrees and 125 degrees.
2. **Stretch.** Make `table_coverage` find the smallest L3 in steps of 0.01 m that fully covers the table with a joint limit of 180 degrees.
3. **Explain.** Explain why a fixed tool direction leaves a hole in the workspace but a free tool direction does not.

## Assessment
- **A. Conceptual.** A two-link arm with L1 = 0.30 m and L2 = 0.25 m has a hole of radius 0.05 m in its workspace. Why does adding a 0.15 m third link remove the hole, if the gripper's direction is free?
- **B. Numerical.** Compute the reachable area of a two-link arm with L1 = 0.40 m and L2 = 0.20 m, and its smallest and largest radius.
- **C. Graph and simulation interpretation.** From the graph of reachable fraction against joint limit, with L = (0.30, 0.25, 0.15) m, how much of the disk is lost at +-90 degrees, and what is the smallest 5 degree step of the limit that keeps 90 percent?
- **D. Design and optimisation challenge.** Design: a work table 0.5 m x 0.3 m has its short edge 0.20 m in front of the base, centred on it. Choose L3 (keeping L1 = 0.30 m and L2 = 0.25 m) and the smallest joint limit in 5 degree steps so every point of the table is reachable. Justify it.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    d = np.radians
    checks.append(("two-link ring area 4 pi L1 L2 = 0.9425 m^2 equals pi (0.55^2 - 0.05^2)",
                   abs(L.annulus_area(0.55, 0.05) - 4 * np.pi * 0.30 * 0.25) < 1e-12 and abs(L.annulus_area(0.55, 0.05) - 0.9425) < 1e-4))
    checks.append(("grid sampling of the two-link ring gives 0.9427 m^2, within 0.2 percent of the formula",
                   abs(L.two_link_area_grid(0.30, 0.25) - L.annulus_area(0.55, 0.05)) < 0.002 * 0.9425))
    checks.append(("hand calculation: L1 = 0.40, L2 = 0.20 gives ring radii 0.60 and 0.20 and area 1.005 m^2",
                   abs(L.annulus_area(0.60, 0.20) - 1.0053) < 1e-3 and abs(L.two_link_area_grid(0.40, 0.20) - 1.0053) < 3e-3))
    checks.append(("forward kinematics: (0, 0, 0) gives (0.70, 0)", np.allclose(L.fk([0, 0, 0]), [0.70, 0.0])))
    q = L.solutions_for_tool((0.45, 0.20), 0.0)
    checks.append(("worked example: tool angle 0 gives (-9.6, 98.6, -89.0) degrees and (77.0, -98.6, 21.7) degrees",
                   len(q) == 2 and np.allclose(np.degrees(q[0]), [-9.59, 98.63, -89.04], atol=0.02)
                   and np.allclose(np.degrees(q[1]), [76.97, -98.63, 21.66], atol=0.02)))
    checks.append(("worked example: cos(q2) = -0.15 for the wrist at (0.30, 0.20)", abs(np.cos(q[0][1]) + 0.15) < 1e-9))
    ok = all(np.allclose(L.fk(s), [0.45, 0.20], atol=1e-9) and abs(L.wrap(sum(s))) < 1e-9 for s in q)
    checks.append(("both solutions put the gripper on the target pointing along +x", bool(ok)))
    checks.append(("a point beyond 0.70 m has no solution and a point at 0.70 m exactly has one",
                   L.solutions_for_tool((0.75, 0.0), 0.0) == [] and len(L.solutions_for_tool((0.70, 0.0), 0.0)) == 1))
    fam = L.solution_family((0.45, 0.20))
    ph = np.degrees([f[0] for f in fam])
    checks.append(("redundancy: 84 arm shapes for one target at 5 degree tool steps, tool directions -80 to 125 degrees",
                   len(fam) == 84 and abs(ph.min() + 80) < 1e-6 and abs(ph.max() - 125) < 1e-6
                   and all(np.allclose(L.fk(f[1:]), [0.45, 0.20], atol=1e-9) for f in fam)))
    checks.append(("with joint limits +-120 degrees only 67 of those shapes remain",
                   len(L.solution_family((0.45, 0.20), limit=d(120))) == 67))
    full = L.reachable_fraction(180)
    checks.append(("no limits: the whole disk of radius 0.70 m (area 1.539 m^2) is reachable",
                   full > 0.999 and abs(np.pi * 0.7 ** 2 - 1.5394) < 1e-3))
    f120, f90 = L.reachable_fraction(120), L.reachable_fraction(90)
    checks.append(("limits +-120 degrees keep 0.892 of the disk and +-90 degrees keep 0.613",
                   abs(f120 - 0.892) < 0.003 and abs(f90 - 0.613) < 0.003))
    checks.append(("limits +-60 degrees keep only 0.246 of the disk", abs(L.reachable_fraction(60) - 0.246) < 0.003))
    checks.append(("limits +-120 degrees: reachable area 1.373 m^2", abs(f120 * np.pi * 0.7 ** 2 - 1.373) < 0.006))
    f125 = L.reachable_fraction(125)
    checks.append(("125 degrees is the first 5-degree step that reaches 0.9 of the disk (0.918)",
                   f120 < 0.9 < f125 and abs(f125 - 0.918) < 0.003))
    checks.append(("longer third link helps: 0.17 m gives 0.906 at +-120 degrees, 0.15 m gives 0.892",
                   abs(L.reachable_fraction(120, np.array([0.30, 0.25, 0.17])) - 0.9064) < 0.003))
    checks.append(("far table corner is 0.716 m from the base", abs(np.hypot(0.70, 0.15) - 0.7159) < 1e-3))
    c70 = L.table_coverage(np.array([0.30, 0.25, 0.15]))
    l72 = np.array([0.30, 0.25, 0.17])
    checks.append(("table coverage: total 0.70 m reaches 0.956 of the grid points, total 0.72 m reaches all",
                   abs(c70 - 0.956) < 0.002 and L.table_coverage(l72) == 1.0))
    checks.append(("with L3 = 0.17 m the table is fully covered at +-105 degrees but not at +-100 (0.974) or +-90 (0.839)",
                   L.table_coverage(l72, 105) == 1.0 and abs(L.table_coverage(l72, 100) - 0.974) < 0.002
                   and abs(L.table_coverage(l72, 90) - 0.839) < 0.002))
    checks.append(("wrap puts angles in (-pi, pi]", abs(L.wrap(3 * np.pi) - np.pi) < 1e-12 and abs(L.wrap(-np.pi) - np.pi) < 1e-12))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
